In [ ]:
#!/usr/bin/env python3
"""
Plot mean RMSD with standard-deviation bands from triplicate simulations.

Expected NPZ structure:
    data: dictionary containing mean, std, and frames for each system
    meta: dictionary containing plotting metadata, such as colors

Example:
    python plot_rmsd.py \
        --input rmsd_analysis_triplicates_na.npz \
        --output-prefix results/figures/rmsd_na_parallel \
        --xlim 0 100 \
        --ylim 0 14
"""

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Plot RMSD means and standard-deviation bands."
    )

    parser.add_argument(
        "--input",
        required=True,
        type=Path,
        help="Input NPZ file generated by the RMSD calculation script.",
    )

    parser.add_argument(
        "--output-prefix",
        required=True,
        type=Path,
        help="Output path without file extension.",
    )

    parser.add_argument(
        "--timestep-fs",
        type=float,
        default=2.0,
        help="MD timestep in femtoseconds. Default: 2.0.",
    )

    parser.add_argument(
        "--dcd-frequency",
        type=int,
        default=1000,
        help="Number of MD steps between saved DCD frames. Default: 1000.",
    )

    parser.add_argument(
        "--xlim",
        nargs=2,
        type=float,
        metavar=("XMIN", "XMAX"),
        default=(0.0, 100.0),
        help="X-axis range in ns. Default: 0 100.",
    )

    parser.add_argument(
        "--ylim",
        nargs=2,
        type=float,
        metavar=("YMIN", "YMAX"),
        default=(0.0, 14.0),
        help="Y-axis range in Å. Default: 0 14.",
    )

    parser.add_argument(
        "--title",
        default=None,
        help="Optional figure title.",
    )

    parser.add_argument(
        "--legend-location",
        default="upper left",
        help="Matplotlib legend location. Default: upper left.",
    )

    parser.add_argument(
        "--dpi",
        type=int,
        default=300,
        help="PNG output resolution. Default: 300.",
    )

    parser.add_argument(
        "--no-show",
        action="store_true",
        help="Save the figure without displaying it.",
    )

    return parser.parse_args()


def configure_matplotlib():
    """設定適合論文圖表的 Matplotlib 格式。"""

    plt.rcParams["axes.linewidth"] = 1.8
    plt.rcParams["xtick.major.width"] = 1.8
    plt.rcParams["ytick.major.width"] = 1.8

    # PDF與PS中的文字保留為可編輯字型
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42


def load_rmsd_data(input_file):
    """
    載入 RMSD NPZ。

    注意：目前資料以 Python dictionary 儲存在 NPZ 中，
    因此需要 allow_pickle=True。只應讀取可信任的 NPZ 檔案。
    """

    if not input_file.exists():
        raise FileNotFoundError(
            f"找不到 RMSD 資料檔：{input_file}"
        )

    with np.load(
        input_file,
        allow_pickle=True,
    ) as loaded:

        required_keys = {"data", "meta"}
        missing_keys = required_keys - set(loaded.files)

        if missing_keys:
            raise KeyError(
                "NPZ 缺少必要欄位："
                + ", ".join(sorted(missing_keys))
            )

        rmsd_data = loaded["data"].item()
        system_meta = loaded["meta"].item()

    if not isinstance(rmsd_data, dict):
        raise TypeError("NPZ 中的 data 必須是 dictionary。")

    if not isinstance(system_meta, dict):
        raise TypeError("NPZ 中的 meta 必須是 dictionary。")

    return rmsd_data, system_meta


def validate_system_data(label, stats):
    """檢查單一系統的 RMSD 資料。"""

    required_fields = {"mean", "std"}

    missing_fields = required_fields - set(stats)

    if missing_fields:
        print(
            f"[警告] 跳過 {label}，缺少："
            f"{', '.join(sorted(missing_fields))}"
        )
        return None

    mean_values = np.asarray(
        stats["mean"],
        dtype=float,
    )

    std_values = np.asarray(
        stats["std"],
        dtype=float,
    )

    if mean_values.ndim != 1 or std_values.ndim != 1:
        print(f"[警告] 跳過 {label}：mean/std 必須是一維陣列。")
        return None

    frame_count = min(
        len(mean_values),
        len(std_values),
    )

    if frame_count == 0:
        print(f"[警告] 跳過 {label}：資料為空。")
        return None

    if len(mean_values) != len(std_values):
        print(
            f"[警告] {label} 的 mean 與 std 長度不同，"
            f"裁切至 {frame_count} frames。"
        )

    return (
        mean_values[:frame_count],
        std_values[:frame_count],
        frame_count,
    )


def get_system_color(label, system_meta):
    """從 meta 取得顏色；未設定時交給 Matplotlib自動選色。"""

    metadata = system_meta.get(label, {})

    if not isinstance(metadata, dict):
        return None

    return metadata.get("color")


def plot_rmsd(
    rmsd_data,
    system_meta,
    time_per_frame_ns,
    args,
):
    """繪製所有系統的平均 RMSD 與 Mean ± SD。"""

    fig, ax = plt.subplots(
        figsize=(12, 8),
        dpi=150,
    )

    plot_count = 0

    print("正在繪製 RMSD：")

    for label, stats in rmsd_data.items():
        validated = validate_system_data(
            label,
            stats,
        )

        if validated is None:
            continue

        mean_values, std_values, frame_count = validated

        time_axis = (
            np.arange(frame_count, dtype=float)
            * time_per_frame_ns
        )

        color = get_system_color(
            label,
            system_meta,
        )

        line = ax.plot(
            time_axis,
            mean_values,
            color=color,
            label=label,
            linewidth=2.6,
            alpha=1.0,
        )[0]

        # 若 meta 沒有指定顏色，取得 Matplotlib自動選擇的顏色
        fill_color = line.get_color()

        ax.fill_between(
            time_axis,
            mean_values - std_values,
            mean_values + std_values,
            color=fill_color,
            alpha=0.20,
            linewidth=0,
        )

        print(
            f"  {label}: {frame_count} frames, "
            f"{time_axis[-1]:.3f} ns"
        )

        plot_count += 1

    if plot_count == 0:
        plt.close(fig)
        raise RuntimeError("沒有有效 RMSD 資料可以繪製。")

    format_axes(ax, args)

    return fig


def format_axes(ax, args):
    """設定論文圖表格式。"""

    if args.title:
        ax.set_title(
            args.title,
            fontsize=30,
            fontweight="bold",
            pad=20,
        )

    ax.set_xlabel(
        "Time (ns)",
        fontsize=30,
        labelpad=12,
    )

    ax.set_ylabel(
        "RMSD (Å)",
        fontsize=30,
        labelpad=12,
    )

    ax.set_xlim(args.xlim)
    ax.set_ylim(args.ylim)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=24,
        length=6,
        width=1.8,
    )

    ax.legend(
        loc=args.legend_location,
        fontsize=20,
        frameon=True,
        framealpha=0.9,
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.8,
        alpha=0.4,
    )


def save_figure(fig, output_prefix, dpi):
    """同時輸出 PNG 與 PDF。"""

    output_prefix.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    png_path = output_prefix.with_suffix(".png")
    pdf_path = output_prefix.with_suffix(".pdf")

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    print(f"Saved PNG: {png_path}")
    print(f"Saved PDF: {pdf_path}")


def main():
    """主程式。"""

    args = parse_arguments()

    configure_matplotlib()

    # 2 fs × 1000 steps = 2000 fs = 0.002 ns/frame
    time_per_frame_ns = (
        args.timestep_fs
        * args.dcd_frequency
        / 1_000_000.0
    )

    print(
        f"Time per frame: "
        f"{time_per_frame_ns:.6f} ns/frame"
    )

    rmsd_data, system_meta = load_rmsd_data(
        args.input
    )

    fig = plot_rmsd(
        rmsd_data=rmsd_data,
        system_meta=system_meta,
        time_per_frame_ns=time_per_frame_ns,
        args=args,
    )

    fig.tight_layout()

    save_figure(
        fig=fig,
        output_prefix=args.output_prefix,
        dpi=args.dpi,
    )

    if args.no_show:
        plt.close(fig)
    else:
        plt.show()


if __name__ == "__main__":
    main()